# Met Antiquities Data Exploration
This notebook provides a simple EDA interface to explore the ingested dataset from `data/metadata_index.parquet`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from IPython.display import Image, display

sns.set_theme(style="darkgrid")

In [ ]:
# Load the metadata file
df = pd.read_parquet('data/metadata_index.parquet')
print(f"Total artifacts: {len(df)}")
df.head()

In [ ]:
# Let's look at the distribution of Culture
plt.figure(figsize=(10, 6))
df['Culture'].value_counts().head(10).plot(kind='bar', color='teal')
plt.title('Top 10 Cultures in the Dataset')
plt.ylabel('Number of Artifacts')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Display a sample image and its serialized text
sample = df.sample(1).iloc[0]
print(f"Title: {sample['Title']}")
print(f"Serialized Text: {sample['text_serialized']}")

object_id = sample['Object ID']
print(f"Fetching image for Object ID: {object_id}...")
# The static Met CSV does not contain raw image URLs directly, so we query the public API
try:
    api_url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{object_id}"
    res = requests.get(api_url, timeout=5)
    if res.status_code == 200:
        data = res.json()
        image_url = data.get('primaryImageSmall') or data.get('primaryImage')
        if image_url:
            print(f"Asset Link: {image_url}")
            display(Image(url=image_url, width=400))
        else:
            print("The API returned no public primary image for this artifact.")
    else:
        print(f"Failed to access Met API (Status {res.status_code}).")
except Exception as e:
    print(f"Error fetching image: {e}")